In [ ]:
# =============================================================================
#   IMPROVED QUANTUM HEART DISEASE CLASSIFIER
#   Multiple strategies for better convergence
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZFeatureMap, TwoLocal
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import COBYLA
from qiskit.quantum_info import SparsePauliOp

# --------------------------- 1. Load & Preprocess Data ---------------------------
FILE_NAME = 'Heart_data.csv'
df = pd.read_csv(FILE_NAME)

FEATURE_COLUMNS = ['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate', 'QuantumPatternFeature']
X_raw = df[FEATURE_COLUMNS].values
y = df['HeartDisease'].values.astype(int)

# Standardize features (zero mean, unit variance) - often better for quantum circuits
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Map to [0, 2π] for angle encoding
X = (X_scaled - X_scaled.min()) / (X_scaled.max() - X_scaled.min()) * 2 * np.pi

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class distribution → No Disease (0): {np.sum(y==0)}, Disease (1): {np.sum(y==1)}")
print(f"Balance: {np.sum(y==1)/len(y)*100:.1f}% disease\n")

# --------------------------- 2. Train/Test Split ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {len(X_train)} | Test: {len(X_test)}\n")

num_qubits = X.shape[1]

# --------------------------- 3. Simplified Quantum Circuit ---------------------------
# Use simpler, more trainable circuit architecture
feature_map = ZFeatureMap(feature_dimension=num_qubits, reps=1)

# TwoLocal with fewer reps for easier optimization
ansatz = TwoLocal(
    num_qubits=num_qubits,
    rotation_blocks=['ry', 'rz'],
    entanglement_blocks='cz',
    entanglement='linear',
    reps=2,
    insert_barriers=False
)

qc = QuantumCircuit(num_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

print(f"Circuit Info:")
print(f"  - Qubits: {qc.num_qubits}")
print(f"  - Trainable params: {ansatz.num_parameters}")
print(f"  - Depth: {qc.depth()}\n")

# --------------------------- 4. Multiple Observable Strategy ---------------------------
# Use multiple observables and combine them (better signal)
observables = [
    SparsePauliOp(['Z' + 'I'*(num_qubits-1)], coeffs=[1.0]),  # First qubit
    SparsePauliOp(['I'*(num_qubits-1) + 'Z'], coeffs=[1.0]),  # Last qubit
    SparsePauliOp(['Z'*num_qubits], coeffs=[1.0])              # All qubits parity
]

estimator = StatevectorEstimator()

# --------------------------- 5. Fast Expectation Computation ---------------------------
def compute_expectations_batch(X_batch, theta):
    """Compute expectations for all samples in batch efficiently"""
    circuits = []
    obs_list = []
    
    # Bind parameters for each sample
    for x_sample in X_batch:
        feature_params = dict(zip(feature_map.parameters, x_sample))
        ansatz_params = dict(zip(ansatz.parameters, theta))
        all_params = {**feature_params, **ansatz_params}
        bound_qc = qc.assign_parameters(all_params)
        
        # Use all three observables
        for obs in observables:
            circuits.append(bound_qc)
            obs_list.append(obs)
    
    # Run all circuits at once (much faster)
    job = estimator.run(list(zip(circuits, obs_list)))
    results = job.result()
    
    # Extract and average expectations
    expectations = []
    for i in range(len(X_batch)):
        # Average the three observable measurements
        exp_vals = [results[i*3 + j].data.evs for j in range(3)]
        avg_exp = np.mean(exp_vals)
        expectations.append(avg_exp)
    
    return np.array(expectations)

def predict_probabilities(X_batch, theta):
    """Convert expectations to probabilities"""
    expectations = compute_expectations_batch(X_batch, theta)
    # Map [-1, 1] to [0, 1]
    probabilities = (1 - expectations) / 2
    return probabilities

# --------------------------- 6. Improved Loss Function ---------------------------
def cost_function(theta):
    """Binary cross-entropy with regularization"""
    probs = predict_probabilities(X_train, theta)
    probs = np.clip(probs, 1e-10, 1 - 1e-10)
    
    # Cross-entropy loss
    ce_loss = -np.mean(y_train * np.log(probs) + (1 - y_train) * np.log(1 - probs))
    
    # Add small L2 regularization to prevent overfitting
    l2_reg = 0.001 * np.sum(theta**2) / len(theta)
    
    total_loss = ce_loss + l2_reg
    return total_loss

# --------------------------- 7. Training with Progress Tracking ---------------------------
print("="*60)
print("TRAINING QUANTUM CLASSIFIER")
print("="*60)

np.random.seed(42)
initial_theta = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)

losses = []
iteration_count = [0]
best_loss = [float('inf')]

def tracked_cost_function(theta):
    """Track and display optimization progress"""
    loss = cost_function(theta)
    losses.append(loss)
    iteration_count[0] += 1
    
    if loss < best_loss[0]:
        best_loss[0] = loss
    
    if iteration_count[0] % 5 == 0:
        print(f"Iter {iteration_count[0]:3d} | Loss: {loss:.4f} | Best: {best_loss[0]:.4f}")
    
    return loss

print("\nOptimizing... (target: loss < 0.4 for good accuracy)\n")
optimizer = COBYLA(maxiter=200, tol=1e-6)

result = optimizer.minimize(
    fun=tracked_cost_function,
    x0=initial_theta
)

optimal_theta = result.x
final_loss = result.fun

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"  - Iterations: {result.nfev}")
print(f"  - Final loss: {final_loss:.4f}")
print(f"  - Best loss: {best_loss[0]:.4f}")
print(f"{'='*60}\n")

# --------------------------- 8. Evaluation ---------------------------
print("Evaluating model...\n")

# Training accuracy
train_probs = predict_probabilities(X_train, optimal_theta)
train_preds = (train_probs >= 0.5).astype(int)
train_acc = accuracy_score(y_train, train_preds)

# Test accuracy
test_probs = predict_probabilities(X_test, optimal_theta)
test_preds = (test_probs >= 0.5).astype(int)
test_acc = accuracy_score(y_test, test_preds)

print(f"{'='*60}")
print(f"FINAL RESULTS")
print(f"{'='*60}")
print(f"Training Accuracy: {train_acc*100:.2f}%")
print(f"Test Accuracy:     {test_acc*100:.2f}%")
print(f"{'='*60}\n")

print("Classification Report (Test Set):")
print(classification_report(y_test, test_preds, 
                          target_names=['No Disease', 'Disease'],
                          digits=3))

# --------------------------- 9. Visualizations ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss
axes[0].plot(losses, linewidth=2.5, color='darkblue')
axes[0].axhline(y=0.4, color='red', linestyle='--', linewidth=2, label='Target (0.4)')
axes[0].axhline(y=0.693, color='orange', linestyle='--', linewidth=2, label='Random (0.693)')
axes[0].set_xlabel('Iteration', fontsize=12, weight='bold')
axes[0].set_ylabel('Loss (Cross-Entropy)', fontsize=12, weight='bold')
axes[0].set_title('Training Loss Curve', fontsize=14, weight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Confusion Matrix
cm = confusion_matrix(y_test, test_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

im = axes[1].imshow(cm_norm, cmap='RdPu', vmin=0, vmax=100)

for i in range(2):
    for j in range(2):
        text_color = "white" if cm_norm[i, j] > 60 else "black"
        axes[1].text(j, i, f"{cm[i, j]}\n({cm_norm[i, j]:.1f}%)",
                    ha="center", va="center",
                    color=text_color, fontsize=18, weight='bold')

axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['No Disease', 'Disease'], fontsize=11)
axes[1].set_yticklabels(['No Disease', 'Disease'], fontsize=11)
axes[1].set_xlabel('Predicted', fontsize=12, weight='bold')
axes[1].set_ylabel('Actual', fontsize=12, weight='bold')
axes[1].set_title(f'Confusion Matrix\nTest Accuracy: {test_acc*100:.2f}%', 
                 fontsize=14, weight='bold')

plt.colorbar(im, ax=axes[1], label='Percentage (%)')
plt.tight_layout()
plt.show()

# Show probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (probs, y_true, title) in enumerate([
    (train_probs, y_train, 'Training Set'),
    (test_probs, y_test, 'Test Set')
]):
    axes[idx].hist(probs[y_true==0], bins=20, alpha=0.6, label='No Disease (0)', color='blue')
    axes[idx].hist(probs[y_true==1], bins=20, alpha=0.6, label='Disease (1)', color='red')
    axes[idx].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold')
    axes[idx].set_xlabel('Predicted Probability', fontsize=12, weight='bold')
    axes[idx].set_ylabel('Count', fontsize=12, weight='bold')
    axes[idx].set_title(f'{title} - Probability Distribution', fontsize=13, weight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
if test_acc >= 0.88:
    print("✓ Target accuracy (88%+) ACHIEVED!")
elif test_acc >= 0.75:
    print("⚠ Decent accuracy, but below target")
else:
    print("✗ Low accuracy - data may not be quantum-separable")
print("="*60)

Dataset: 500 samples, 6 features
Class distribution → No Disease (0): 200, Disease (1): 300
Balance: 60.0% disease

Training: 400 | Test: 100

Circuit Info:
  - Qubits: 6
  - Trainable params: 36
  - Depth: 2

TRAINING QUANTUM CLASSIFIER

Optimizing... (target: loss < 0.4 for good accuracy)

Iter   5 | Loss: 0.7053 | Best: 0.7050
